In [1]:
from datetime import datetime
from doctr.io import DocumentFile
from doctr.models import ocr_predictor

In [2]:
model = ocr_predictor(pretrained=True)

def extract_text(ocr_model, image_file_path: str):
    """Extrae el texto de una imagen a partir de un modelo OCR"""

    doc = DocumentFile.from_images(image_file_path)
    result = ocr_model(doc)
    return result

def show_ocr_result(result):
    """Muestra una imagen de resultado y el texto extraido"""

    # mostrar resultado
    #result.show()
    lista = []
    # mostrar texto con mas de 50% de confianza
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                words = []
                for word in line.words:
                    if word.confidence > 0.5:
                        words.append(word.value)
                lista.append(' '.join(words))

    return lista

In [3]:
def extraccion_antiguo(words_ante):
    info = {}

    for i in range(len(words_ante)):
        if len(words_ante[i]) == 9 and words_ante[i].isalnum():
            if words_ante[i][:8].isdigit() and words_ante[i][8:].isalpha():
                info['dni'] = words_ante[i]

        elif 'APEL' in words_ante[i]:
            info['apellido1'] = words_ante[i+1]

            if words_ante[i+2] != 'NOMBRE/NOM' and words_ante[i+2] != 'NOMBRE':
                info['apellido2'] = words_ante[i+2]

        elif 'NOM' in words_ante[i]:
            if ' ' in words_ante[i+1]:
                info['nombre1'] = words_ante[i+1].split()[0]
                info['nombre2'] = words_ante[i+1].split()[1]
            else:
                info['nombre1'] = words_ante[i+1]

        elif 'FECHA' in words_ante[i] and not words_ante[i+1].isalpha():
            info['f_nacimiento'] = words_ante[i+1]

        elif words_ante[i] == 'VALIDEZ/VALIDESA' or words_ante[i] == 'VALI':
            if len(words_ante[i+1]) == 20 and not words_ante[i+1].isalpha():
                aux = words_ante[i+1][:9]
                if not aux.isalpha():
                    info['num_sop'] = aux
                    info['f_validez'] = words_ante[i+1][10:]

    return info

In [4]:
def extraccion_moderno(words_ante):
    info = {}

    for i in range(len(words_ante)):
        if len(words_ante[i]) == 9 and words_ante[i].isalnum():
            if words_ante[i][:8].isdigit() and words_ante[i][8:].isalpha():
                info['dni'] = words_ante[i]
            else:
                if words_ante[i][3:].isdigit() and words_ante[i][:2].isalpha():
                    info['num_sop'] = words_ante[i]

        elif 'APEL' in words_ante[i]:
            info['apellido1'] = words_ante[i+1]

            if words_ante[i+2] != 'NOMBRE/NOM' and words_ante[i+2] != 'NOMBRE':
                info['apellido2'] = words_ante[i+2]

        elif 'NOM' in words_ante[i]:
            if ' ' in words_ante[i+1]:
                info['nombre1'] = words_ante[i+1].split()[0]
                info['nombre2'] = words_ante[i+1].split()[1]
            else:
                info['nombre1'] = words_ante[i+1]

        elif words_ante[i] == 'ESP' and not words_ante[i+1].isalpha():
            info['f_nacimiento'] = words_ante[i+1]

        elif words_ante[i] == 'VALIDEZ/VALIDESA' or words_ante[i] == 'VALI':
            if len(words_ante[i+1]) == 10 and not words_ante[i+1].isalpha():
                info['f_emision'] = words_ante[i+1]
                info['f_validez'] = words_ante[i+2]
            
            elif len(words_ante[i+1]) == 21 and not words_ante[i+1].isalpha():
                partes = words_ante[i+1].split()

                fecha = []
                fecha.append(partes[0])
                fecha.append(partes[1])
                fecha.append(partes[2])
                info['f_emision'] = ' '.join(fecha)

                fecha = []
                fecha.append(partes[3])
                fecha.append(partes[4])
                fecha.append(partes[5])
                info['f_validez'] = ' '.join(fecha)

    return info

In [5]:
def val_digitos(info):
    letra = ['T', 'R', 'W', 'A', 'G', 'M', 'Y', 'F', 'P', 'D', 'X', 'B', 'N', 'J', 'Z', 'S', 'Q', 'V', 'H', 'L', 'C', 'K', 'E']
    dni = info['dni']
    dig = int(dni[:8])
    let = dni[8:]
    aux = dig%23

    if let != letra[aux]:
        print('DNI NO VALIDO, VUELVA A INTENTARLO')
        

In [ ]:
def val_fechas(info):
    fecha1_str = info['f_emision']
    fecha1 = datetime.strptime(fecha1_str, "%d %m %Y")
    fecha2_str = info['f_nacimiento']  
    fecha2 = datetime.strptime(fecha2_str, "%d %m %Y")
    fecha3_str = info['f_validez'] 
    fecha3 = datetime.strptime(fecha3_str, "%d %m %Y")
    fecha4 = datetime.now()

    edad = fecha1.year - fecha2.year
    if (fecha1.month, fecha1.day) < (fecha2.month, fecha2.day):
        edad -= 1

    if fecha3 < fecha4:
        print('DNI NO VALIDO, VUELVA A INTENTARLO')
        
    renov = fecha3.year - fecha1.year

    if edad < 5 and renov == 2:
        return
    elif edad < 30 and renov == 5:
        return
    elif edad < 70 and renov == 10:
        return
    else:
        print('DNI NO VALIDO, VUELVA A INTENTARLO')

In [7]:
def procesar_filas(words_reve, info):


    fila1, fila2, fila3 = words_reve[-3:]


    info_fila1 = [
        fila1[:5],
        fila1[5:14],
        fila1[14:15],
        fila1[15:24],
        fila1[25:]      
    ]

    n = len(fila2)

    info_fila2 = [
        fila2[:6],
        fila2[6:8],
        fila2[8:14],
        fila2[14:15],
        fila2[15:18],
        fila2[18:n-1],
        fila2[n-1:]
    ]

    claves = []
    for clave, valor in info.items():
        claves.append(clave)

    try:
        ap1 = len(info["apellido1"])
        ap2 = len(info["apellido2"])
        n1  = len(info["nombre1"])
        if 'nombre2' in claves:
            n2  = len(info['nombre2'])  
        else: 
            n2 = 0
        
    except KeyError as e:
        raise ValueError(f"Falta la clave en info: {e}")

    n_total = n1 + n2  

    pos = 0

    parte1 = fila3[pos:pos+ap1]
    pos += ap1

    parte2 = fila3[pos:pos+1]
    pos += 1

    parte3 = fila3[pos:pos+ap2]
    pos += ap2

    parte4 = fila3[pos:pos+2]
    pos += 2

    if n2 != 0:

        parte5 = fila3[pos:pos+n1]
        pos += n1
    
        parte6 = fila3[pos:pos+1]
        pos += 1

        parte7 = fila3[pos:]

        info_fila3 = [
            parte1,
            parte2,
            parte3,
            parte4,
            parte5, 
            parte6,
            parte7
        ]

    else:
        parte5 = fila3[pos:pos+n_total]
        pos += n_total

        parte6 = fila3[pos:]

        info_fila3 = [
            parte1,
            parte2,
            parte3,
            parte4,
            parte5, 
            parte6
        ]

    return info_fila1, info_fila2, info_fila3

In [8]:
def formatear_fecha(valor):
        grupos = [valor[i:i+2] for i in range(0, len(valor), 2)]
        return ' '.join(grupos[::-1])
    
    
def limpiar(valor):
    return valor.replace('<', '').strip()


def construir_info_reverso(info_fila1, info_fila2, info_fila3):
    
    f_nac = formatear_fecha(info_fila2[0])
    f_val = formatear_fecha(info_fila2[2])

    info_reverso = {
        'num_sop': limpiar(info_fila1[1]),
        'dni': limpiar(info_fila1[3]),
        'f_nacimiento': f_nac,
        'f_validez': f_val,
        'apellido1': limpiar(info_fila3[0]),
        'apellido2': limpiar(info_fila3[2]),
        'nombre1': limpiar(info_fila3[4]),
    }

    posible_nombre2 = limpiar(info_fila3[6]) if len(info_fila3) > 6 else ""

    if posible_nombre2:
        info_reverso['nombre2'] = posible_nombre2

    return info_reverso

In [9]:
def normalizar_fecha(fecha):
    if not fecha or len(fecha) < 8:
        return fecha
    return fecha[:2] + fecha[2:6] + fecha[8:]


def comparar_info(info, info_reverso, verbose=True):

    resultados = {}

    comparaciones = {
        "dni": ("El dni", lambda a, b: a == b),
        "num_sop": ("El num_sop", lambda a, b: a == b),
        "f_nacimiento": ("La fecha de nacimiento",
                         lambda a, b: normalizar_fecha(a) == b),
        "f_validez": ("La fecha de validez",
                      lambda a, b: normalizar_fecha(a) == b),
        "apellido1": ("El primer apellido", lambda a, b: a == b),
        "apellido2": ("El segundo apellido", lambda a, b: a == b),
        "nombre1": ("El primer nombre", lambda a, b: a == b),
    }

    for clave, (texto, comparador) in comparaciones.items():
        v1 = info.get(clave)
        v2 = info_reverso.get(clave)

        if v1 is None or v2 is None:
            resultados[clave] = None
            if verbose:
                print(f"{texto}: no se pudo comparar (dato faltante)")
            continue

        coincide = comparador(v1, v2)
        resultados[clave] = coincide

        if verbose:
            print(f"{texto} {'coincide' if coincide else 'no coincide'}")

    v1 = info.get("nombre2")
    v2 = info_reverso.get("nombre2")

    if not v1 and not v2:
        resultados["nombre2"] = True
        if verbose:
            print("El segundo nombre coincide (no presente en ambos)")
    elif v1 and v2:
        coincide = v2 in v1
        resultados["nombre2"] = coincide
        if verbose:
            print(f"El segundo nombre {'coincide' if coincide else 'no coincide'}")
    else:
        resultados["nombre2"] = None
        if verbose:
            print("El segundo nombre: no se pudo comparar (solo presente en uno)")

    return resultados

In [10]:
def validar_dni(img1, img2):
    text = extract_text(model, img1)
    words_ante = show_ocr_result(text)
    
    text = extract_text(model, img2)
    words_reve = show_ocr_result(text)

    reino = False
    for i in range(len(words_ante)):
        if 'REINO' in words_ante[i]:
            reino = True

    if reino:  
        info = extraccion_moderno(words_ante)
        val_fechas(info)
    else:
        info = extraccion_antiguo(words_ante)

    val_digitos(info)
            
    info_fila1, info_fila2, info_fila3 = procesar_filas(words_reve, info)
    info_reverso = construir_info_reverso(info_fila1, info_fila2, info_fila3)

    resultados = comparar_info(info, info_reverso)


In [11]:
img1 = './DNI_especimen/pepe_anverso.jpeg'
img2 = './DNI_especimen/pepe_reverso.jpeg'

validar_dni(img1, img2)

El dni coincide
El num_sop coincide
La fecha de nacimiento coincide
La fecha de validez coincide
El primer apellido coincide
El segundo apellido coincide
El primer nombre coincide
El segundo nombre coincide


In [14]:
img1 = './DNI_especimen/hugo_anverso.jpeg'
img2 = './DNI_especimen/hugo_reverso.jpeg'

validar_dni(img1, img2)

AttributeError: type object 'datetime.datetime' has no attribute 'format'

In [ ]:
img1 = './DNI_especimen/pedro_anverso.jpeg'
img2 = './DNI_especimen/pedro_reverso.jpeg'

validar_dni(img1, img2)

El dni coincide
El num_sop coincide
La fecha de nacimiento coincide
La fecha de validez coincide
El primer apellido coincide
El segundo apellido coincide
El primer nombre coincide
El segundo nombre coincide


In [ ]:
img1 = './DNI_especimen/ma_anverso.jpeg'
img2 = './DNI_especimen/ma_reverso.jpeg'

validar_dni(img1, img2)

El dni coincide
El num_sop coincide
La fecha de nacimiento coincide
La fecha de validez coincide
El primer apellido coincide
El segundo apellido coincide
El primer nombre coincide
El segundo nombre coincide
